## То же самое на TensorFlow (Keras)



In [7]:
import os

import time
import certifi
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

os.environ["SSL_CERT_FILE"] = certifi.where()

print("using device:", "GPU" if tf.config.list_physical_devices("GPU") else "CPU")  
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

(tf_train_x, tf_train_y), (tf_test_x, tf_test_y) = keras.datasets.mnist.load_data()
tf_train_x = (tf_train_x.astype("float32") / 255.0 - MNIST_MEAN) / MNIST_STD
tf_test_x = (tf_test_x.astype("float32") / 255.0 - MNIST_MEAN) / MNIST_STD

using device: GPU


In [8]:

tf_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(512),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(1024),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(512),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(10),
])

tf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)


In [9]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.total_start = time.time()

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        print(f"epoch {epoch + 1} took {time.time() - self.epoch_start:.2f}s")

    def on_train_end(self, logs=None):
        print(f"total training time: {time.time() - self.total_start:.2f}s")



In [10]:
tf_model.fit(
    tf_train_x, tf_train_y,
    validation_data=(tf_test_x, tf_test_y),
    batch_size=512,
    epochs=10,
    callbacks=[EpochTimer()],
)

Epoch 1/10
118/118 [==============================] - 5s 25ms/step - loss: 0.3663 - accuracy: 0.8875 - val_loss: 0.1866 - val_accuracy: 0.9420
Epoch 2/10
118/118 [==============================] - 2s 21ms/step - loss: 0.1494 - accuracy: 0.9547 - val_loss: 0.1002 - val_accuracy: 0.9713
Epoch 3/10
118/118 [==============================] - 2s 21ms/step - loss: 0.1116 - accuracy: 0.9658 - val_loss: 0.0906 - val_accuracy: 0.9723
Epoch 4/10
118/118 [==============================] - 2s 21ms/step - loss: 0.0938 - accuracy: 0.9708 - val_loss: 0.0823 - val_accuracy: 0.9756
Epoch 5/10
118/118 [==============================] - 2s 21ms/step - loss: 0.0765 - accuracy: 0.9757 - val_loss: 0.0741 - val_accuracy: 0.9781
Epoch 6/10
118/118 [==============================] - 2s 21ms/step - loss: 0.0647 - accuracy: 0.9801 - val_loss: 0.0661 - val_accuracy: 0.9809
Epoch 7/10
118/118 [==============================] - 2s 21ms/step - loss: 0.0610 - accuracy: 0.9801 - val_loss: 0.0604 - val_accuracy: 0.9827

In [11]:
tf_model.save("mnist_mlp_tf.keras")
print("model saved to mnist_mlp_tf.keras")

model saved to mnist_mlp_tf.keras


## Сравнение с CPU

In [12]:
with tf.device('/CPU:0'):
    MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

    (tf_train_x, tf_train_y), (tf_test_x, tf_test_y) = keras.datasets.mnist.load_data()
    tf_train_x = (tf_train_x.astype("float32") / 255.0 - MNIST_MEAN) / MNIST_STD
    tf_test_x = (tf_test_x.astype("float32") / 255.0 - MNIST_MEAN) / MNIST_STD

    tf_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(512),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(1024),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(512),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(10),
    ])

    tf_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )
    
    class EpochTimer(keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.total_start = time.time()

        def on_epoch_begin(self, epoch, logs=None):
            self.epoch_start = time.time()

        def on_epoch_end(self, epoch, logs=None):
            print(f"epoch {epoch + 1} took {time.time() - self.epoch_start:.2f}s")

        def on_train_end(self, logs=None):
            print(f"total training time: {time.time() - self.total_start:.2f}s")
        
    tf_model.fit(
    tf_train_x, tf_train_y,
    validation_data=(tf_test_x, tf_test_y),
    batch_size=512,
    epochs=10,
    callbacks=[EpochTimer()],)

Epoch 1/10
118/118 [==============================] - 7s 47ms/step - loss: 0.3711 - accuracy: 0.8842 - val_loss: 0.1858 - val_accuracy: 0.9451
Epoch 2/10
118/118 [==============================] - 5s 45ms/step - loss: 0.1489 - accuracy: 0.9539 - val_loss: 0.1237 - val_accuracy: 0.9643
Epoch 3/10
118/118 [==============================] - 5s 45ms/step - loss: 0.1101 - accuracy: 0.9668 - val_loss: 0.0867 - val_accuracy: 0.9739
Epoch 4/10
118/118 [==============================] - 5s 45ms/step - loss: 0.0910 - accuracy: 0.9720 - val_loss: 0.0828 - val_accuracy: 0.9766
Epoch 5/10
118/118 [==============================] - 5s 45ms/step - loss: 0.0761 - accuracy: 0.9770 - val_loss: 0.0672 - val_accuracy: 0.9799
Epoch 6/10
118/118 [==============================] - 5s 45ms/step - loss: 0.0656 - accuracy: 0.9796 - val_loss: 0.0706 - val_accuracy: 0.9796
Epoch 7/10
118/118 [==============================] - 6s 48ms/step - loss: 0.0585 - accuracy: 0.9814 - val_loss: 0.0691 - val_accuracy: 0.9807